# 🧪 Toxicity Prediction — ML Pipeline
### Target: Predict **Toxic** vs **NonToxic** molecules
---
**Pipeline Steps:**
1. Load & Inspect Data
2. Exploratory Data Analysis (EDA)
3. Data Preprocessing
4. Feature Selection
5. Model Training with Cross-Validation
6. Results & Evaluation

## 📦 Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully")

## 📂 Cell 2 — Load & Inspect Data

In [ ]:
df = pd.read_csv('data-1.csv')

print("=" * 50)
print(f"  Dataset Shape  : {df.shape}")
print(f"  Total Features : {df.shape[1] - 1}")
print(f"  Total Samples  : {df.shape[0]}")
print("=" * 50)

print("\n📋 First 5 rows:")
df.head()

## 🎯 Cell 3 — Target Class Distribution

In [ ]:
class_counts = df['Class'].value_counts()

print("Class Distribution:")
print("-" * 30)
for cls, cnt in class_counts.items():
    bar = "█" * int(cnt / 2)
    print(f"  {cls:12s}: {cnt:4d}  ({cnt/len(df)*100:.1f}%)  {bar}")

print(f"\nClass imbalance ratio: {class_counts.max() / class_counts.min():.2f}:1")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
palette = {"Toxic": "#E74C3C", "NonToxic": "#2ECC71"}

# Bar chart
bars = axes[0].bar(class_counts.index, class_counts.values,
                   color=[palette[c] for c in class_counts.index],
                   edgecolor='white', width=0.5)
axes[0].set_title("Class Count", fontweight='bold', fontsize=13)
axes[0].set_ylabel("Count")
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(int(bar.get_height())), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%',
            colors=[palette[c] for c in class_counts.index],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title("Class Proportion", fontweight='bold', fontsize=13)

plt.suptitle("Target Variable Distribution", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔍 Cell 4 — Missing Values & Data Types

In [ ]:
X_raw = df.drop('Class', axis=1)

print("Missing Values Check:")
print("-" * 30)
total_missing = df.isnull().sum().sum()
print(f"  Total missing values : {total_missing}")
print(f"  Missing features     : {(df.isnull().sum() > 0).sum()}")

print("\nData Types:")
print("-" * 30)
for dtype, count in X_raw.dtypes.value_counts().items():
    print(f"  {str(dtype):12s}: {count} features")

print("\nBasic Statistics (sample):")
X_raw.describe().T.head(10)

## 📊 Cell 5 — Feature Variance Analysis

In [ ]:
var = X_raw.var()
n_zero    = (var == 0).sum()
n_low_var = (var < 0.01).sum()
n_normal  = (var >= 0.01).sum()

print("Variance Analysis:")
print("-" * 40)
print(f"  Total features          : {len(var)}")
print(f"  Constant (var = 0)      : {n_zero}")
print(f"  Near-zero  (var < 0.01) : {n_low_var}")
print(f"  Normal     (var >= 0.01): {n_normal}")

# Plot variance distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

var_nonzero = var[var > 0]
axes[0].hist(np.log10(var_nonzero + 1e-10), bins=50,
             color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title("Feature Variance Distribution (log₁₀)", fontweight='bold')
axes[0].set_xlabel("log₁₀(Variance)")
axes[0].set_ylabel("Number of Features")
axes[0].axvline(np.log10(0.01), color='red', linestyle='--',
                linewidth=1.5, label='Threshold (0.01)')
axes[0].legend()

# Categories
categories = ['Constant\n(var=0)', 'Near-Zero\n(var<0.01)', 'Normal\n(var≥0.01)']
counts_v = [n_zero, n_low_var - n_zero, n_normal]
colors_v = ['#E74C3C', '#F39C12', '#2ECC71']
axes[1].bar(categories, counts_v, color=colors_v, edgecolor='white', width=0.5)
axes[1].set_title("Feature Categories by Variance", fontweight='bold')
axes[1].set_ylabel("Count")
for i, v in enumerate(counts_v):
    axes[1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 🔗 Cell 6 — Correlation Analysis

In [ ]:
corr_matrix = X_raw.corr().abs()
upper_tri = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

thresholds = [0.7, 0.8, 0.9, 0.95, 0.99]
print("High Correlation Pairs:")
print("-" * 35)
for t in thresholds:
    count = (upper_tri > t).sum().sum()
    print(f"  r > {t} : {count:4d} pairs")

# Encode target
le = LabelEncoder()
y = le.fit_transform(df['Class'])
print(f"\nLabel Encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Top features correlation with target
corr_with_target = X_raw.apply(lambda col: abs(col.corr(pd.Series(y))))
top_corr = corr_with_target.nlargest(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top feature correlations with target
colors_c = plt.cm.RdYlGn(np.linspace(0.2, 0.9, 15))
axes[0].barh(top_corr.index[::-1], top_corr.values[::-1],
             color=colors_c, edgecolor='white')
axes[0].set_title("Top 15 Features — Correlation with Target", fontweight='bold')
axes[0].set_xlabel("Absolute Correlation")

# Correlation heatmap of top 10
top10 = top_corr.index[:10].tolist()
corr_top = X_raw[top10].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, mask=mask, cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, annot=True, fmt='.2f',
            linewidths=0.5, ax=axes[1], annot_kws={"size": 7})
axes[1].set_title("Correlation Heatmap — Top 10 Features", fontweight='bold')
axes[1].tick_params(axis='x', rotation=45, labelsize=7)
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.show()

## 📈 Cell 7 — Feature Distributions (Toxic vs NonToxic)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

selector_eda = SelectKBest(f_classif, k=9)
selector_eda.fit(X_raw, y)
scores_eda = pd.Series(selector_eda.scores_, index=X_raw.columns)
top9 = scores_eda.nlargest(9).index.tolist()

print("Top 9 Features by ANOVA F-Score:")
print("-" * 45)
for i, feat in enumerate(top9, 1):
    print(f"  {i}. {feat:<35s} F={scores_eda[feat]:.2f}")

fig, axes = plt.subplots(3, 3, figsize=(14, 11))
axes = axes.flatten()
palette = {"Toxic": "#E74C3C", "NonToxic": "#2ECC71"}

for i, feat in enumerate(top9):
    for cls in ['Toxic', 'NonToxic']:
        data = df[df['Class'] == cls][feat]
        axes[i].hist(data, bins=25, alpha=0.6, label=cls,
                     color=palette[cls], edgecolor='white')
    axes[i].set_title(f"{feat}\nF={scores_eda[feat]:.2f}",
                      fontsize=9, fontweight='bold')
    axes[i].legend(fontsize=7)
    axes[i].set_ylabel("Count")

plt.suptitle("Top 9 Feature Distributions — Toxic vs NonToxic",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 Cell 8 — Data Preprocessing

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

X = X_raw.copy()
print("Preprocessing Steps:")
print("=" * 50)
print(f"  Starting features : {X.shape[1]}")

# Step 1: Remove constant features
vt1 = VarianceThreshold(threshold=0)
X = pd.DataFrame(vt1.fit_transform(X), columns=X.columns[vt1.get_support()])
print(f"\n  Step 1 — Remove constant features")
print(f"           Removed : {X_raw.shape[1] - X.shape[1]}")
print(f"           Remaining : {X.shape[1]}")

# Step 2: Remove near-zero variance
vt2 = VarianceThreshold(threshold=0.01)
X = pd.DataFrame(vt2.fit_transform(X), columns=X.columns[vt2.get_support()])
removed_nzv = X_raw.shape[1] - X.shape[1]
print(f"\n  Step 2 — Remove near-zero variance (< 0.01)")
print(f"           Removed : {removed_nzv}")
print(f"           Remaining : {X.shape[1]}")

# Step 3: Remove highly correlated features
corr3 = X.corr().abs()
upper3 = corr3.where(np.triu(np.ones(corr3.shape), k=1).astype(bool))
to_drop = [col for col in upper3.columns if any(upper3[col] > 0.95)]
X.drop(columns=to_drop, inplace=True)
print(f"\n  Step 3 — Remove highly correlated (r > 0.95)")
print(f"           Removed : {len(to_drop)}")
print(f"           Remaining : {X.shape[1]}")

# Step 4: Standard Scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
print(f"\n  Step 4 — StandardScaler applied")
print(f"           Mean ≈ 0, Std ≈ 1")
print("=" * 50)
print(f"\n✅ Final shape after preprocessing: {X_scaled.shape}")

# Visualise reduction
stages = ['Original', 'After Var\nFilter', 'After Corr\nFilter', 'After\nScaling']
counts_s = [X_raw.shape[1], X_raw.shape[1] - removed_nzv,
            X_raw.shape[1] - removed_nzv - len(to_drop),
            X_raw.shape[1] - removed_nzv - len(to_drop)]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(stages, counts_s,
              color=['#3498DB','#F39C12','#E74C3C','#2ECC71'],
              edgecolor='white', width=0.5)
ax.set_title("Feature Count at Each Preprocessing Stage", fontweight='bold', fontsize=13)
ax.set_ylabel("Number of Features")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(int(bar.get_height())), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 🎯 Cell 9 — Feature Selection (ANOVA F-Test)

In [ ]:
K_FEATURES = 50

selector = SelectKBest(f_classif, k=K_FEATURES)
X_selected = selector.fit_transform(X_scaled, y)
selected_features = X_scaled.columns[selector.get_support()].tolist()

f_scores = pd.Series(
    selector.scores_[selector.get_support()],
    index=selected_features
).sort_values(ascending=False)

print(f"✅ Selected top {K_FEATURES} features via ANOVA F-Test")
print(f"   Reduced from {X_scaled.shape[1]} → {K_FEATURES} features\n")

print("Top 15 Selected Features:")
print("-" * 45)
for i, (feat, score) in enumerate(f_scores.head(15).items(), 1):
    bar = "█" * int(score / 0.5)
    print(f"  {i:2d}. {feat:<35s} F={score:.2f}  {bar}")

# Plot
fig, ax = plt.subplots(figsize=(11, 8))
colors_f = plt.cm.plasma(np.linspace(0.1, 0.9, 20))
top20_f = f_scores.head(20)
ax.barh(top20_f.index[::-1], top20_f.values[::-1], color=colors_f, edgecolor='white')
ax.set_title("Top 20 Features by ANOVA F-Score\n(Selected for Modelling)",
             fontweight='bold', fontsize=13)
ax.set_xlabel("F-Score")
for i, (val, name) in enumerate(zip(top20_f.values[::-1], top20_f.index[::-1])):
    ax.text(val + 0.05, i, f'{val:.2f}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 🤖 Cell 10 — Model Training with 5-Fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Random Forest"       : RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    "Gradient Boosting"   : GradientBoostingClassifier(n_estimators=150, random_state=42),
    "Logistic Regression" : LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "SVM (RBF)"           : SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
}

results = {}
print("Training models with 5-Fold Stratified Cross-Validation...")
print("=" * 60)

for name, model in models.items():
    cv_res = cross_validate(
        model, X_selected, y, cv=cv,
        scoring=['accuracy','f1','roc_auc','precision','recall'],
        return_train_score=False
    )
    results[name] = {
        'accuracy' : cv_res['test_accuracy'],
        'f1'       : cv_res['test_f1'],
        'roc_auc'  : cv_res['test_roc_auc'],
        'precision': cv_res['test_precision'],
        'recall'   : cv_res['test_recall'],
    }
    print(f"\n  [{name}]")
    print(f"  {'Metric':<12} {'Mean':>8}  {'Std':>8}")
    print(f"  {'-'*30}")
    for metric in ['accuracy','f1','roc_auc','precision','recall']:
        m = results[name][metric].mean()
        s = results[name][metric].std()
        print(f"  {metric:<12} {m:>8.4f}  {s:>8.4f}")

best_name = max(results, key=lambda n: results[n]['roc_auc'].mean())
print(f"\n{'=' * 60}")
print(f"★  Best Model → {best_name}  (ROC-AUC = {results[best_name]['roc_auc'].mean():.4f})")

## 📊 Cell 11 — Model Comparison Chart

In [ ]:
model_names = list(results.keys())
metrics     = ['accuracy', 'f1', 'roc_auc']
m_labels    = ['Accuracy', 'F1-Score', 'ROC-AUC']
colors_m    = ['#3498DB', '#E74C3C', '#2ECC71']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Grouped bar chart
x = np.arange(len(model_names))
w = 0.25
for i, (metric, label) in enumerate(zip(metrics, m_labels)):
    means = [results[n][metric].mean() for n in model_names]
    stds  = [results[n][metric].std()  for n in model_names]
    axes[0].bar(x + i*w, means, w, yerr=stds, label=label,
                capsize=4, color=colors_m[i], alpha=0.85, edgecolor='white')
axes[0].set_xticks(x + w)
axes[0].set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=9)
axes[0].set_ylim(0.3, 1.05)
axes[0].set_ylabel("Score")
axes[0].set_title("Model Comparison — All Metrics", fontweight='bold', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].axhline(0.9, color='gray', linestyle='--', linewidth=0.8, label='0.9 line')

# Boxplot ROC-AUC
plot_data = {n: results[n]['roc_auc'] for n in model_names}
bp = axes[1].boxplot(plot_data.values(),
                     labels=[n.replace(' ', '\n') for n in plot_data],
                     patch_artist=True,
                     medianprops=dict(color='white', linewidth=2.5))
box_colors = ['#3498DB','#E74C3C','#2ECC71','#9B59B6']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1].set_ylabel("ROC-AUC")
axes[1].set_title("ROC-AUC Distribution (5-Fold CV)", fontweight='bold', fontsize=13)
axes[1].axhline(0.9, color='gray', linestyle='--', linewidth=0.8)
axes[1].tick_params(axis='x', labelsize=9)

plt.suptitle("Cross-Validation Model Evaluation", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 📉 Cell 12 — ROC Curves (Best Model per Fold)

In [ ]:
best_model = models[best_name]
tprs, aucs_list = [], []
mean_fpr = np.linspace(0, 1, 100)

fig, ax = plt.subplots(figsize=(8, 6))

for fold, (tr, te) in enumerate(cv.split(X_selected, y)):
    best_model.fit(X_selected[tr], y[tr])
    probs = best_model.predict_proba(X_selected[te])[:, 1]
    fpr, tpr, _ = roc_curve(y[te], probs)
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    aucs_list.append(roc_auc_score(y[te], probs))
    ax.plot(fpr, tpr, alpha=0.3, linewidth=1.2, color='#3498DB',
            label=f'Fold {fold+1} (AUC={aucs_list[-1]:.3f})' if fold < 5 else "")

mean_tpr = np.mean(tprs, axis=0)
ax.plot(mean_fpr, mean_tpr, color='#E74C3C', linewidth=2.5,
        label=f'Mean ROC (AUC={np.mean(aucs_list):.3f} ± {np.std(aucs_list):.3f})')
ax.fill_between(mean_fpr,
                np.mean(tprs,0) - np.std(tprs,0),
                np.mean(tprs,0) + np.std(tprs,0),
                alpha=0.15, color='#3498DB', label='±1 std dev')
ax.plot([0,1],[0,1],'k--', linewidth=1, label='Random Classifier')
ax.set_title(f"ROC Curves — {best_name}\n(5-Fold Stratified CV)",
             fontweight='bold', fontsize=13)
ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 🏆 Cell 13 — Feature Importance (Best Model)

In [ ]:
best_model.fit(X_selected, y)

if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_,
                    index=selected_features).sort_values(ascending=False)
    imp_label = "Feature Importance (Gini)"
else:
    from sklearn.inspection import permutation_importance
    perm = permutation_importance(best_model, X_selected, y,
                                  n_repeats=15, random_state=42)
    imp = pd.Series(perm.importances_mean,
                    index=selected_features).sort_values(ascending=False)
    imp_label = "Permutation Importance"

top20_imp = imp.head(20)
print(f"Top 20 Important Features ({imp_label}):")
print("-" * 55)
for i, (feat, val) in enumerate(top20_imp.items(), 1):
    bar = "█" * int(val * 300)
    print(f"  {i:2d}. {feat:<35s} {val:.4f}  {bar}")

fig, ax = plt.subplots(figsize=(10, 8))
c_imp = plt.cm.viridis(np.linspace(0.15, 0.9, 20))
ax.barh(top20_imp.index[::-1], top20_imp.values[::-1],
        color=c_imp, edgecolor='white')
ax.set_title(f"Top 20 Feature Importances\n{best_name} — {imp_label}",
             fontweight='bold', fontsize=13)
ax.set_xlabel(imp_label)
plt.tight_layout()
plt.show()

## ✅ Cell 14 — Final Summary

In [ ]:
print("=" * 60)
print("  TOXICITY PREDICTION PIPELINE — FINAL SUMMARY")
print("=" * 60)
print(f"  Dataset         : {df.shape[0]} samples, {df.shape[1]-1} features")
print(f"  Classes         : Toxic ({(df['Class']=='Toxic').sum()}) | NonToxic ({(df['Class']=='NonToxic').sum()})")
print()
print("  Preprocessing:")
print(f"    Original features    : {X_raw.shape[1]}")
print(f"    After preprocessing  : {X_scaled.shape[1]}")
print(f"    After feat. selection: {K_FEATURES}")
print()
print("  Cross-Validation : StratifiedKFold (k=5)")
print()
print(f"  {'Model':<25} {'Accuracy':>9} {'F1':>9} {'ROC-AUC':>9}")
print("  " + "-" * 55)
for name in model_names:
    acc = results[name]['accuracy'].mean()
    f1  = results[name]['f1'].mean()
    auc = results[name]['roc_auc'].mean()
    star = " ★ BEST" if name == best_name else ""
    print(f"  {name:<25} {acc:9.4f} {f1:9.4f} {auc:9.4f}{star}")

print("=" * 60)
print(f"\n  ★  Best Model  →  {best_name}")
print(f"     ROC-AUC      →  {results[best_name]['roc_auc'].mean():.4f}")
print(f"     F1-Score     →  {results[best_name]['f1'].mean():.4f}")
print(f"     Accuracy     →  {results[best_name]['accuracy'].mean():.4f}")
print("=" * 60)